In [ ]:
### developing schemas

In [24]:
from typing import List
from pydantic import BaseModel, Field

import pydantic
import enum
import os
import logging
from copy import deepcopy
import glob
import pathlib
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely as shpy

import joblib

In [ ]:
import pydantic

In [ ]:
df_loc = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\data\processed\processed_data.csv'
df = pd.read_csv( df_loc )
df.describe()


In [ ]:
df.columns

In [ ]:
def load_us_states():
    states_file = pathlib.Path('us_states.txt')
    state_ls = states_file.read_text().splitlines()
    state_dictn = {   e_state.replace(' ', '_'): e_state  for e_state in state_ls   }
    return state_dictn

USState = enum.Enum( 'USState', load_us_states(), type= str )


In [ ]:
filepath = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\api\us_states.txt'

states_file = pathlib.Path( filepath )
states_ls = states_file.read_text().splitlines()

state_dictn = { e_state.replace( ' ', '_'): e_state  for e_state in states_ls }

USState = enum.Enum( 'USState', state_dictn, type= str )





In [ ]:
class SourceType( str, enum.Enum ):
    gas = 'gas'
    oil = 'oil'
    coal = 'coal'
    other_fossil = 'other_fossil'
    biomass = 'biomass'
    waste = 'waste'

class Emission_Prediction_request( pydantic.BaseModel ):
     capacity: float = Field( ..., ge= 0, description= 'Capacity of the Industry - must be non negative numerical value' )
     capacity_factor: float = Field( ..., ge= 0, description= 'Capacity factor of the Industry - must be non negative numerical value' )
     activity: float = Field( ..., ge= 0, description= 'Activity of the Industry - must be non negative numerical value' )
     source_type: SourceType
     state: USState
     area: float = Field( ..., ge= 0, description= 'Area of the State - must be non negative numerical value' )
     pop: int = Field( ..., ge= 0, description= 'Population of the State - must be non negative integer value' )


class Emission_Prediction_response( pydantic.BaseModel ):
     predicted_emission: float 
     confidence_interval: List[ float ]
     feature_importance: dict
     prediction_time: str


In [ ]:
df['source_type'].unique()

In [ ]:
### inference

In [1]:
import joblib

import pathlib

import pandas as pd
import numpy as np

## local lib
import features._create_features


In [25]:
MODEL_PATH = 'models/trained/greenhouse_emission_predict_model.pkl' 
MODEL_PATH = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\models\trained\greenhouse_emission_predict_model.pkl'

PREPROCESSOR_PATH = 'models/trained/preprocessor.pkl'
PREPROCESSOR_PATH = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\models\trained\preprocessor.pkl'

In [26]:
model = joblib.load( MODEL_PATH )
preprocessor = joblib.load( PREPROCESSOR_PATH )

prediction_Response = {
    'capacity': 1200.5,
    'capacity_factor': 0.62,
    'activity': 850000.0,
    'source_type': 'gas',
    'state': 'Texas',
    'area': 695662.0,
    'pop2020': 30145505
}



In [ ]:
asd = set( {1, 2, 3} )

asd

In [27]:
# input_df = pd.DataFrame( [ prediction_Response.dict() ] )
input_df = pd.DataFrame( [prediction_Response] )
# print( f'\n{input_df}\n' )

xx = features._create_features.main( input_df )


## pandas show all columns
pd.set_option('display.max_columns', None)
xx

NameError: name 'features' is not defined

In [ ]:
## features remaining after one-hot encoding categorical variables
REMAINING_Features_ls = [        
    'capacity', 'capacity_factor', 'activity',
    'area', 'pop2020',
    'log1p_activity', 'log1p_capacity', 'log1p_pop2020', 'log1p_area',
    'log1Pop_density', 'activity_per_capita', 'activity_per_area',
    'capacity_per_capita', 'capacity_density', 'potential_output',
    'utilization_ratio', 'activity_capacityFactor', 'activity_per_capacity',
    'activity_capacity', 'capacity_factor_capacity',
]
xx_remainingFeatures_df = xx.filter( REMAINING_Features_ls )
xx_remainingFeatures_df

## Fit and transform (OHE) the data
x_transformed = preprocessor.fit_transform( xx )

In [ ]:
x_arr = x_transformed.toarray() if hasattr( x_transformed, 'toarray' )  else x_transformed
xCat_transformed_df = pd.DataFrame(x_arr)  ## <-- only catorical features after transformation
xCat_transformed_df


In [ ]:
{
  'source_types': [
    'biomass',
    'coal',
    'gas',
    'oil',
    'other_fossil',
    'waste'
  ]
}


In [ ]:
## get one-hot encoded feature names
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
ohe_names = ohe.get_feature_names_out(preprocessor.transformers_[0][2])

print( ohe_names )

print( f'''The last numbbbber of the categorical-trasnformed column is --> {len(xCat_transformed_df.columns)-1}
Thus we begin naming xx_remainingFeatures_df starting from  --> {len(xCat_transformed_df.columns)}''' )


##### create _features



In [1]:
import json
import logging
from copy import deepcopy
from pathlib import Path
import pathlib

import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path
import json

def _load_domain_values( pathlib_path, key: str) -> set[str]:
    """
    Load a set of string values from a JSON file.

    Expected JSON structure:
      { "<key>": [ ... ] }

    Notes:
    - `path_str` must be a (Pathlib) path to a JSON file
    """
    obj = json.loads( pathlib_path.read_text(encoding='utf-8') )
    return { str(x) for x in obj[key] }


In [19]:
source_dir = pathlib.Path( 'src' )
_DOMAIN_DIR = source_dir / 'domain'

_SOURCE_TYPES_JSON = _DOMAIN_DIR / 'source_types.json'
_USSTATES_JSON = _DOMAIN_DIR / 'usstates.json'
_USSTATES_JSON_path =  pathlib.Path( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\usstates.json'  )
_SOURCE_TYPE_JSON_path =  pathlib.Path( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\source_types.json'  )
_USSTATES_JSON


WindowsPath('src/domain/usstates.json')

In [20]:
states = _load_domain_values( _USSTATES_JSON_path, key= 'states' )
sourceType = _load_domain_values( _SOURCE_TYPE_JSON_path, key= 'source_types' )

sourceType

{'biomass', 'coal', 'gas', 'oil', 'other_fossil', 'waste'}

In [28]:
df = pd.read_csv( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\data\processed\processed_data.csv' )
df.head(2)

,start_time,end_time,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,modified_date,source_type,state,area,pop2020
0,2022-01-01,2022-12-31,214000.0,0.513,138.0,0.345,417000.0,2023-11-01 10:00:00,gas,Alabama,12.899239,5024279.0
1,2019-01-01,2019-12-31,204000.0,0.513,138.0,0.329,398000.0,2023-11-01 10:00:00,gas,Alabama,12.899239,5024279.0


In [ ]:
# df_copy['state'] = _canon_state( df_copy['state'] )
# df_copy['source_type'] = _canon_source_type( df_copy['source_type'] )

In [31]:
df['state'].str.strip().str.replace( ' ', '', regex= False )


0        Alabama
1        Alabama
2        Alabama
3        Alabama
4          Texas
          ...   
8844    Michigan
8845    Michigan
8846    Michigan
8847    Michigan
8848    Michigan
Name: state, Length: 8849, dtype: object